# Lesson 1.5 — From linear regression to neural networks (and what BERT actually is)

Companion to [Lesson 1](lesson_01_linear_regression.ipynb).

After Lesson 1 you've trained a model with **2 parameters** that learns to fit a line. After Lesson 4 you train a tiny BERT with **3000 parameters** that does fill-in-the-blank. This notebook bridges the gap.

We'll answer two questions that should be on your mind:

1. **"Wait — is BERT just linear regression with more knobs? Or is it a different kind of model?"**
2. **"When do you need a neural network instead of linear regression?"**

The story we'll build today, on a single dataset:

| Approach | Parameters | Can it fit the curve? |
|---|---|---|
| **Linear regression** (L1 — `y = w*x + b`) | 2 | ❌ No |
| **Polynomial regression** (same loop, more features) | 3 | ✅ Yes (with handcrafted x² feature) |
| **Neural network** (Linear → ReLU → Linear) | ~30 | ✅ Yes (no feature engineering needed) |
| **Transformer** (BERT, PRAGMA) | thousands–billions | ✅ Yes, on sequences |

> 🔑 **The big takeaway:** the **5-line training loop is the same for all four**. What changes is the **model in the middle**. BERT and PRAGMA are NEURAL NETWORKS — specifically a kind called Transformers — NOT linear regression. The training recipe is shared; the model architecture is different.


## Step 0 — Imports

In [ ]:
import torch                                                  # Load PyTorch — the deep learning library providing tensors (n-dim arrays), autograd (automatic gradient computation), neural network modules, and gradient-based optimisers we'll use throughout this entire notebook.
import torch.nn as nn                                         # Import PyTorch's "neural network" submodule. Contains building-block classes like nn.Linear, nn.Embedding, nn.ReLU, plus the nn.Module base class every model inherits from.
import torch.nn.functional as F                               # Import the "functional" forms of common operations (F.relu, F.softmax, F.mse_loss). Same math as the nn.* classes but stateless — convention is to alias as F for brevity.

torch.manual_seed(0)                                          # Fix PyTorch's random number generator with seed 0. Every random op (tensor init, sampling, etc.) becomes reproducible — your printed numbers will match this notebook's exactly on every run.
torch.set_printoptions(precision=3, sci_mode=False)           # Configure tensor printing: 3 decimal places, no scientific notation. Makes output readable (0.123 instead of 1.23e-01) and keeps printed tables aligned for visual inspection.

## Step 1 — A new dataset: this one is a CURVE

In Lesson 1 the data was a straight line: `y = 2x + 1`. Linear regression nailed it because a line is exactly what linear regression can fit.

This time the secret rule is **a parabola**: `y = x² - 4x + 3`. Let's see if linear regression can still handle it.

In [ ]:
# Generate the data
x_data = torch.linspace(-2, 6, 40)                                        # torch.linspace creates 40 evenly-spaced numbers from -2 to 6 (both inclusive). Returns a 1D tensor of shape (40,) — these are our input x values covering the parabola's domain.
y_true = x_data ** 2 - 4 * x_data + 3 + torch.randn(40) * 0.5             # Compute the secret rule y = x² - 4x + 3, then add Gaussian noise (mean 0, std 0.5) via torch.randn. Mimics realistic data which is never perfectly clean.

print(f"x range: {x_data.min().item():.1f} to {x_data.max().item():.1f}")  # .min() returns a 0-dim tensor; .item() extracts the Python float for f-string formatting. Without .item() the print would include extra tensor metadata.
print(f"y range: {y_true.min().item():.1f} to {y_true.max().item():.1f}")  # Same pattern for y. Shows the y values span roughly -1 to +15 — that's the height range of the parabola plus noise.
print()                                                                    # Blank line for visual separation in the output. Pure formatting convenience, no logic.

# Show first/last few points
print(f"{'x':>6}  {'y_true':>7}")                                          # Print a header row. ':>6' means right-align in a 6-char field, ':>7' similarly. Creates a clean aligned column layout for the table below.
print("-" * 16)                                                            # Print 16 dashes as a horizontal separator under the header — simple ASCII technique for visually distinguishing the header from the data rows.
for i in [0, 5, 10, 15, 20, 25, 30, 35, 39]:                               # Loop over 9 representative indices spanning the dataset (not all 40, that would be too many). Choosing 0 and 39 ensures we see both endpoints.
    print(f"{x_data[i].item():>6.2f}  {y_true[i].item():>7.2f}")           # Print one (x, y) pair per iteration. ':>6.2f' = right-aligned float, 6 chars wide, 2 decimal places. Aligning makes patterns visible.

**Look at the y values.** They go DOWN and then UP. That's not a line. No straight line could possibly fit all those points well — a line either goes up the whole way or down the whole way.

Let's draw it (ASCII):

In [ ]:
def ascii_scatter(x, y, marker='*', width=60, height=20, title=""):  # Define a tiny helper that plots (x, y) points in ASCII without matplotlib. width/height set plot size; marker is the character drawn at each point.
    if title:                                                          # Print the title line if provided. Lets us label the plot from the caller without needing a separate print statement.
        print(title)                                                   # Output the title above the plot. Standard formatting move for any chart so the reader knows what they're looking at.
    xmin, xmax = x.min().item(), x.max().item()                        # Compute x-axis range from the data. .min/.max return tensor scalars; .item() converts to Python floats so we can do arithmetic below.
    ymin, ymax = y.min().item(), y.max().item()                        # Same for y. We'll use these to scale every data point into grid coordinates so the plot fills the available space.
    grid = [[" "] * width for _ in range(height)]                      # Build a 2D list of spaces — our blank canvas. Each row is a list of `width` single-character strings. Using lists (mutable) so we can write into them.
    for xi, yi in zip(x.tolist(), y.tolist()):                          # Walk through paired (x, y) values. zip pairs them up; .tolist() converts tensors to Python lists so the iteration is fast and clean.
        col = int((xi - xmin) / (xmax - xmin) * (width - 1))            # Convert this point's x into a column index (0..width-1). The (xi-xmin)/(xmax-xmin) normalises x to [0,1], then we scale to the grid width.
        row = height - 1 - int((yi - ymin) / (ymax - ymin) * (height - 1))   # Same for y, but flipped: higher y → smaller row index (because text grids count rows downward, opposite of math y-axis).
        if 0 <= col < width and 0 <= row < height:                     # Safety check — only write if the computed (row, col) is inside the grid. Protects against floating-point edge cases at the boundaries.
            grid[row][col] = marker                                    # Stamp the marker character into the grid. After the loop, the grid contains the picture; we just need to print it.
    print(f"{ymax:6.1f} |" + "".join(grid[0]))                          # Print the top row of the grid with the maximum y value as a label. The '|' acts as the y-axis line. Joining grid[0] turns the list into a string.
    for row in grid[1:-1]:                                              # Print all rows except first/last (those have y-value labels). Using slicing grid[1:-1] excludes the endpoints we already printed / will print.
        print("       |" + "".join(row))                                # Each interior row gets a leading 7-space pad to align under the y-axis labels above and below. Just spacing for visual alignment.
    print(f"{ymin:6.1f} |" + "".join(grid[-1]))                         # Print the bottom row of the grid with the minimum y value as a label. Mirrors the top row's structure for visual symmetry.
    print("       +" + "-" * width)                                     # Print the x-axis line: a '+' as the corner connector, then `width` dashes. Creates the bottom border of the plot frame.
    print(f"        x={xmin:.1f}{' ' * (width-12)}x={xmax:.1f}")        # Print x-axis tick labels at both ends. The (width-12) gap centers them; '.1f' rounds to 1 decimal for compact readability.

ascii_scatter(x_data, y_true, title="The data (a parabola):")           # Actually draw the plot. Title goes above; the U-shape of the points should be visible — showing why a single straight line can't fit this data.

See the U-shape? That's our challenge. Linear regression can only draw STRAIGHT lines, and there is no straight line that hits all those points.

## Step 2 — Try linear regression (will fail)

Same model as Lesson 1: `y = w*x + b`. Two parameters. Let's see how badly it fails.

In [ ]:
# Model: y = w*x + b
w = torch.tensor(0.0, requires_grad=True)                                  # Create the slope parameter, initialised to zero. requires_grad=True tells PyTorch to track every operation using w so .backward() can later compute the gradient with respect to it.
b = torch.tensor(0.0, requires_grad=True)                                  # Create the intercept parameter, also at zero. Same requires_grad flag — both w and b are our "knobs". With both at 0, the model initially predicts 0 for every input.
opt = torch.optim.SGD([w, b], lr=0.001)                                    # torch.optim.SGD is Stochastic Gradient Descent — the "nudge each parameter against its gradient" optimiser. Pass the parameter list and learning rate (step size, 0.001 here).

# Train
for _ in range(1000):                                                       # Repeat the 5-line training loop 1000 times. The underscore means "loop counter not used" — purely about repetition count, not about which iteration we're on.
    y_pred = w * x_data + b                                                 # Forward pass: compute the model's prediction for every input at once. Broadcasting expands the scalar w/b across the 40-element x_data tensor.
    loss = ((y_pred - y_true) ** 2).mean()                                  # Mean squared error: subtract truth from predictions, square element-wise, take the average. One scalar measuring how wrong the model is on this batch.
    opt.zero_grad(); loss.backward(); opt.step()                            # Three statements on one line: clear last step's gradients; backpropagate (compute dloss/dw, dloss/db); take one step downhill on w and b. The core of training.

print(f"Linear regression result: y = {w.item():.3f} * x + {b.item():.3f}")  # Print the final learned line. .item() pulls the scalar value out of each 0-dim tensor so f-strings can format them as ordinary floats.
print(f"Final loss: {loss.item():.3f}")                                     # Show the final loss — should be high (~25) because no straight line fits the parabolic data. This is the point we'll improve on in the next sections.
print()                                                                     # Blank line for visual separation. Pure formatting.
print("(Compare to L1 which got loss ≈ 0.0001 on its line.)")               # Reminder of the L1 result for context. L1's data was actually linear, so a line fit perfectly. Here, the data is a curve, so a line fails badly.
print()                                                                     # Another blank line.

# Predictions
y_pred_linear = (w * x_data + b).detach()                                   # Compute predictions one more time and detach from the autograd graph. .detach() means "I'm just using this for printing, don't track gradients here" — saves memory.
print("First few predictions vs truth:")                                    # Header label introducing the prediction inspection table below.
print(f"{'x':>6}  {'y_true':>7}  {'y_pred':>7}  {'error':>7}")              # Column headers, right-aligned in fixed-width fields so the numeric rows line up vertically underneath them.
print("-" * 33)                                                             # Print a horizontal divider line. The 33 was chosen to match the total width of the header above.
for i in [0, 10, 20, 30, 39]:                                                # Walk through 5 representative indices spanning the dataset. We don't print all 40 rows — too much noise. These 5 cover both endpoints and middle.
    err = y_pred_linear[i].item() - y_true[i].item()                         # Compute the prediction error (model minus truth). Positive = overpredicted, negative = underpredicted. Useful diagnostic.
    print(f"{x_data[i].item():>6.2f}  {y_true[i].item():>7.2f}  {y_pred_linear[i].item():>7.2f}  {err:>+7.2f}")    # Print one row of the table. ':+7.2f' adds a sign for clarity; '.2f' rounds to 2 decimals.

**Look at the errors.** At the endpoints (low and high x) the model's predictions are way off. The best straight line just CAN'T fit a curve.

Loss settles around ~3-5, much worse than L1's ~0.0001 on its straight-line data.

## Step 3 — Trick: give linear regression more features

Here's the key insight that confuses everyone: **"linear regression" doesn't mean "fits a line in x"**. It means **"linear in the parameters"**. We can give it any transformation of x as a feature!

If we add `x²` as a feature, the model becomes:

$$y = w_1 \cdot x + w_2 \cdot x^2 + b$$

This is still LINEAR REGRESSION (3 parameters: w₁, w₂, b). But now it can fit a parabola.

This is called **polynomial regression**, but it's mathematically just linear regression with extra features.

In [ ]:
# Same SGD, more parameters
w1 = torch.tensor(0.0, requires_grad=True)                                  # The slope (coefficient of x). Initialised to 0 like before. Still "linear regression" because the model is linear IN THE PARAMETERS, even though it includes x².
w2 = torch.tensor(0.0, requires_grad=True)                                  # The coefficient of x². NEW parameter — handles the curvature. This is the feature we manually engineered to let a linear model fit a parabola.
b  = torch.tensor(0.0, requires_grad=True)                                  # The intercept (bias). Initialised to 0. Same role as in L1's linear regression — shifts the entire curve up or down.
opt = torch.optim.SGD([w1, w2, b], lr=0.0005)                               # SGD over THREE parameters now. Smaller learning rate (0.0005 vs 0.001) because x² values get large (up to 36), so gradients are bigger and we need smaller steps.

# Train
for _ in range(2000):                                                        # Train for 2000 steps (more than the linear model needed). Why more? Because we have more parameters to adjust, and the gradients are scaled differently for w1 vs w2.
    y_pred = w1 * x_data + w2 * x_data**2 + b                                # Forward pass: y = w1*x + w2*x² + b. Three terms summed element-wise across all 40 inputs. This is the "polynomial regression" formula.
    loss = ((y_pred - y_true) ** 2).mean()                                   # Same MSE loss as before. The loss formula doesn't care how many parameters or features we have — it just measures prediction quality.
    opt.zero_grad(); loss.backward(); opt.step()                             # Same 5-line training pattern, on one line for compactness. Zeroes grads, backprops through ALL three parameters now, then updates each.

print(f"Polynomial regression result:")                                      # Header for the final result printout.
print(f"  y = {w1.item():.3f} * x + {w2.item():.3f} * x² + {b.item():.3f}")  # Print the learned formula by extracting each parameter's scalar value with .item() and formatting to 3 decimal places.
print(f"  (true rule was:  y =  -4.000 * x +  1.000 * x² + 3.000)")          # Show what the SECRET rule was for comparison. The model should recover values very close to -4, 1, 3 (within noise).
print()                                                                       # Blank line.
print(f"Final loss: {loss.item():.3f}    (was ~25 for the pure linear model — a 25x improvement)")  # Highlight the improvement. Same training loop, just two more features → 25× lower loss. The model class matters.
print()                                                                       # Another blank line.
y_pred_poly = (w1 * x_data + w2 * x_data**2 + b).detach()                    # Compute final predictions and detach from autograd graph. Stored for later side-by-side comparison with the linear and neural net models.
print("First few predictions vs truth:")                                      # Header for the inspection table.
print(f"{'x':>6}  {'y_true':>7}  {'y_pred':>7}  {'error':>7}")                # Column headers, all right-aligned in fixed widths so the numbers below align cleanly.
print("-" * 33)                                                               # Horizontal divider.
for i in [0, 10, 20, 30, 39]:                                                 # Same five representative indices as before, so results are comparable across the linear/polynomial/NN tables.
    err = y_pred_poly[i].item() - y_true[i].item()                            # Compute the prediction error for this point. Should be small (within noise level ~0.5).
    print(f"{x_data[i].item():>6.2f}  {y_true[i].item():>7.2f}  {y_pred_poly[i].item():>7.2f}  {err:>+7.2f}")  # Print one row of the table — x, truth, prediction, signed error.

**Beautiful.** Loss drops dramatically. The model recovered the true coefficients (almost) perfectly:

- True: `y = -4x + x² + 3`
- Found: `y ≈ -4x + 1x² + 3`

**Same training loop. More features. Better model.** This is still linear regression — we just gave it more knobs and more inputs.

🤔 **But wait — there's a catch.** How would you know to add `x²`? What if the relationship was `sin(x)` or something more complex?

You'd have to TRY many features (x, x², x³, sin(x), e^x...) and see which ones help. For simple data this is doable. For complex data (text, images, banking events) — there are too many possible features to enumerate.

**That's where neural networks come in.**

## Step 4 — Enter the neural network

A neural network is a **stack of linear transformations with nonlinear activations in between**.

The simplest one (a **Multilayer Perceptron**, or MLP) looks like this:

```
input x
   │
   ▼
Linear  ──► 8 hidden numbers
   │
   ▼
ReLU    ──► same 8 numbers, but negative values clipped to 0
   │
   ▼
Linear  ──► 1 output number (the prediction)
```

The key insight: **the ReLU activation introduces NONLINEARITY**. Without it, two stacked linear layers would still produce a linear function (because composing linear functions gives a linear function). With ReLU in between, the network can model curves, kinks, and arbitrarily complex patterns.

**Universal approximation theorem (don't worry about the math):** with enough hidden units, an MLP can approximate ANY smooth function. You don't have to handcraft features like `x²` — the network learns them automatically.

In [ ]:
class MLP(nn.Module):                                                        # Define a custom neural-network class by inheriting from nn.Module — the base class for every PyTorch model. Inheritance gives us .parameters(), .to(device), training/eval modes, etc.
    """A tiny neural network: 1 input → 8 hidden units → 1 output."""     # Docstring describing the architecture. Conventional Python practice — visible via help(MLP) and rendered nicely by IDEs and Jupyter.
    def __init__(self, hidden=8):                                              # The constructor takes one argument: how many hidden units. Default of 8 keeps things small and visualisable. Larger hidden sizes give more capacity but more parameters.
        super().__init__()                                                     # Call the parent nn.Module's __init__ — required boilerplate so PyTorch's internal bookkeeping (parameter registration, etc.) gets initialised before we add our own layers.
        self.layer1 = nn.Linear(1, hidden)     # input → hidden                # First linear layer: takes 1 input feature, produces `hidden` (8) output features. Holds a (8, 1) weight matrix and a (8,) bias vector — 16 trainable numbers total.
        self.layer2 = nn.Linear(hidden, 1)     # hidden → output               # Second linear layer: takes the 8 hidden values, produces 1 output. Holds a (1, 8) weight matrix and a (1,) bias — 9 more trainable numbers. Total: 25 params.

    def forward(self, x):                                                      # Define the forward pass. PyTorch will call this method whenever we do `net(x)`. The forward method is the model's "rule" — what it does with the input.
        h = self.layer1(x)                                                     # Apply the first linear layer. Computes h = x @ W^T + b for each row of x. Output shape (batch, 8) — 8 numbers per input, one per hidden unit.
        h = F.relu(h)                          # the nonlinearity              # Apply ReLU: clip negative values to 0 element-wise. This is THE crucial nonlinearity — without it, stacking Linears collapses to a single Linear and the network has no extra power.
        return self.layer2(h)                                                  # Apply the second linear layer to the post-ReLU hidden values. Output shape (batch, 1) — the final prediction. We don't apply ReLU here because outputs can be any real number.

net = MLP(hidden=8)                                                            # Instantiate the model. PyTorch automatically calls __init__, which creates the two Linear layers with random initial weights (Kaiming-uniform by default).
total = sum(p.numel() for p in net.parameters())                               # Count total trainable parameters. .parameters() yields all weight tensors; .numel() returns the number of elements in each. Summing gives total params (25 here).
print(f"Tiny MLP architecture:")                                                # Header for the layer breakdown printout.
for name, p in net.named_parameters():                                         # Iterate over all trainable tensors with their attribute names. Gives us strings like 'layer1.weight' and the corresponding tensor for inspection.
    shape_str = str(tuple(p.shape))                                            # Convert the tensor's shape to a string. p.shape is a torch.Size; we wrap with tuple() then str() so it formats predictably in the table.
    print(f"  {name:<20s}  shape {shape_str:<10s}  {p.numel()} params")        # Print one row per parameter tensor — left-align name in 20 chars, shape in 10 chars, then the numel count. Clean tabular layout.
print(f"\nTotal parameters: {total}")                                          # Print the grand total. \n adds a blank line first for visual separation. Should equal 8 + 8 + 8 + 1 = 25.
print(f"(Compare to: 2 for linear regression, 3 for polynomial.)")             # Direct comparison to the simpler models from earlier cells. Makes it obvious how much "capacity" we've added by switching model classes.

### 🎨 Visualise the architecture

Before we train, let's draw the network as a wiring diagram. Knowing the SHAPE of the model up front makes it much easier to follow what's happening during training.

In [ ]:
# ASCII architecture diagram of the MLP
hidden = 8                                                                                                  # Number of hidden units we used in the model. Used below for both labeling and looping over the diagram rows.
print("Neural network architecture (Linear → ReLU → Linear):")                                              # Top-level header so the reader knows they're about to see the architecture, not training output.
print()                                                                                                     # Blank spacer line.
print(f"    {'INPUT':<10s}      {'HIDDEN ('+str(hidden)+' units)':<22s}   {'OUTPUT':<10s}")                  # Print the column headers above the diagram. The :<10s and :<22s left-align the labels in fixed-width fields to position the columns.
print()                                                                                                     # Blank spacer line.
# Top of diagram
print(f"                          ┌───┐")                                                                   # Top edge of the hidden-units box. Unicode box-drawing characters render as a clean rectangle in monospace fonts (Jupyter, terminals).
print(f"                          │h₁ │──┐")                                                                # First hidden unit row. The ──┐ shows a horizontal wire branching down toward the output. h₁ uses unicode subscript for h sub 1.
for i in range(2, hidden):                                                                                  # Loop to draw the middle hidden units. range(2, 8) is units 2 through 7 (we draw unit 1 above and unit 8 below explicitly).
    print(f"                          ├───┤  │")                                                            # Box separator between hidden units. ├ and ┤ are connector characters where outer box edges meet horizontal dividers.
    print(f"                          │h_{i}│──┤")                                                          # The middle hidden units. We use h_2, h_3, etc (regular ASCII underscore subscript) rather than h₂ for simplicity in the loop.
print(f"                          ├───┤  │")                                                                # Bottom separator above the last hidden unit. Same logic as the loop separators.
print(f"      x ─────────────────►│h_{hidden}│──┤───────► y")                                               # The bottom hidden unit row, also showing the main flow: input x flows in from the left, prediction y flows out to the right.
print(f"                          └───┘  │")                                                                # Bottom edge of the hidden-units box. Closes the rectangle.
print(f"                                 │ (weighted sum)")                                                 # Annotation reminding the reader that layer2 takes the 8 hidden-unit outputs and computes a weighted sum to get y.
print()                                                                                                     # Spacer line.
print("       layer1: nn.Linear(1, " + str(hidden) + ")        ReLU clipping")                              # Caption identifying which PyTorch component implements which part of the diagram. Uses string concatenation to insert hidden=8.
print("                                  layer2: nn.Linear(" + str(hidden) + ", 1)")                        # Same caption for layer2. The shift in column position aligns it with the second part of the diagram.
print()                                                                                                     # Spacer line.
print("Parameter breakdown:")                                                                               # Header for the parameter inventory table below.
print(f"  layer1.weight  shape (8, 1)    8 numbers   ← one 'slope' per hidden unit")                        # Each hidden unit has its own slope coefficient for the input x — that's 8 numbers in layer1.weight (one per hidden unit).
print(f"  layer1.bias    shape (8,)      8 numbers   ← one 'kink point' per hidden unit")                    # Each hidden unit also has its own bias, which together with the slope determines where the ReLU "kicks in" — the kink point at x = -b/w.
print(f"  layer2.weight  shape (1, 8)    8 numbers   ← how much each hidden unit contributes")              # layer2 has one weight per hidden unit — the coefficient by which that unit's ReLU output is added into the final prediction.
print(f"  layer2.bias    shape (1,)      1 number    ← final offset")                                       # One last bias for the output — shifts the whole prediction curve up or down. Analogous to b in y = mx + b.
print(f"                                ────────")                                                          # Visual separator (just dashes) underlining the parameter count column before we print the total.
print(f"                                  {total} total")                                                   # Print the grand total of parameters using the value we computed earlier. Should be 25.
print()                                                                                                     # Blank spacer.
print("Conceptually:")                                                                                       # Header introducing the intuitive interpretation of the network's parameters and operations.
print("  Each hidden unit h_i computes:  ReLU(w_i · x + b_i)")                                              # The math for a single hidden unit. ReLU clips below zero; above the kink, it's a linear ramp with slope w_i.
print("    → A 'knee' function: zero below some threshold, then linear ramp upward.")                       # Geometric interpretation: ReLU(w*x + b) is a piecewise-linear function with a single bend (kink) — visually like an elbow.
print("    → The knee is at x = -b_i / w_i.")                                                               # Where exactly does the kink occur? When w*x + b = 0, i.e., x = -b/w. This is the "turn-on" point for the unit.
print("  The 8 knees ADD UP (weighted by layer2) to make the final prediction curve.")                       # Layer 2 takes a weighted sum of the 8 knee functions. With enough knees, you can approximate any continuous curve (universal approximation theorem).
print("  With 8 knees you can approximate any reasonable curve.")                                            # The pedagogical punchline: this is why MLPs work for general function approximation, even with simple ReLU units.

### 🎲 What do the weights look like at INITIALISATION (random)?

PyTorch initialises weights randomly. Let's print them. The model is currently incompetent — these numbers have no relationship to the data.

In [ ]:
print("Initial (random) weights:")                                          # Header for the pre-training weight inspection table.
print()                                                                       # Blank line for separation.
print(f"  {'unit':<6s}  {'w (slope)':>11s}  {'b (bias)':>11s}  {'kink at x =':>13s}")  # Column headers, right-aligned where appropriate so numeric columns underneath line up neatly.
print("  " + "-" * 50)                                                        # Horizontal divider line. 50 dashes chosen to roughly match the total column width above.
for i in range(hidden):                                                       # Loop over each of the 8 hidden units. We'll inspect this unit's slope, bias, and the x-position of its ReLU kink.
    w = net.layer1.weight[i, 0].item()                                        # Pull out the i-th hidden unit's slope. layer1.weight has shape (8, 1) — row i is unit i's weight; [i, 0] grabs the single value (input feature 0).
    b = net.layer1.bias[i].item()                                             # Pull out the i-th hidden unit's bias. layer1.bias has shape (8,) — one per hidden unit. .item() converts the 0-dim tensor to a Python float.
    # Where does the ReLU "turn on"? When w*x + b = 0, i.e. x = -b/w
    kink = -b / w if abs(w) > 1e-6 else float('inf')                          # Compute the kink point algebraically. If w is essentially zero, the line is flat — we'd divide by zero, so we use infinity as a sentinel.
    print(f"  h_{i+1:<4d}  {w:>+11.3f}  {b:>+11.3f}  {kink:>+13.2f}")          # Print one row per hidden unit. Use i+1 so units are 1-indexed (h_1..h_8) which feels more natural than 0-indexed in a pedagogical setting.
print()                                                                        # Blank line.
print("Layer 2 (how each hidden unit contributes to the output):")             # Header for layer 2 inspection.
print(f"  layer2 weights: {net.layer2.weight.detach()[0].tolist()}")           # Layer 2's weights as a Python list. layer2.weight has shape (1, 8); [0] grabs the single output row; .tolist() converts to a plain list for printing.
print(f"  layer2 bias:    {net.layer2.bias.item():.3f}")                       # The single layer2 bias (one number). This is the final offset added after summing all hidden-unit contributions.
print()                                                                        # Blank line.
print("Right now these are noise. Training will reshape them so each hidden")  # Educational narration: emphasize that the current values are uninformative — they're literally random samples from a uniform distribution.
print("unit becomes a useful 'feature detector' for some region of x.")        # Foreshadow what training will accomplish — each unit will specialise to a section of the input range, making the model parts work together.

## Step 5 — Train the MLP on the SAME parabola data

Same training loop. Same loss function. Same optimizer. Only the model changed.

In [ ]:
import copy                                                              # Standard library module that provides deep/shallow copy operations. We don't actually use copy here anymore (left in case of future need), since .clone() handles tensor copying for snapshots.

net = MLP(hidden=8)                                                       # Re-instantiate a fresh MLP with random weights. Re-instantiating ensures we start training from scratch rather than continuing from the (already-printed) initial weights.
opt = torch.optim.Adam(net.parameters(), lr=0.05)                         # Use Adam (not SGD). Adam adapts the per-parameter learning rate based on gradient history — handles the very different magnitudes between layer1 and layer2 gradients better than vanilla SGD.

# We need to reshape x_data to (N, 1) — neural nets expect a batch dimension
x_in = x_data.unsqueeze(-1)     # (40, 1)                                  # nn.Linear expects (batch, features). x_data is (40,) — we add a feature axis with unsqueeze(-1), turning it into (40, 1): 40 samples, 1 feature each.
y_target = y_true.unsqueeze(-1) # (40, 1)                                  # Same reshape for the target. F.mse_loss compares element-wise so the shapes must match the model output (which will be (40, 1)).

# Capture weight snapshots at these training steps so we can see evolution
CHECKPOINTS = [0, 100, 500, 1000, 2000]                                    # The training steps at which we'll save a complete weight snapshot. Picked to span "random" → "learning" → "converged" so the evolution is visible.
snapshots = {}                                                             # Dict to store the saved weights, keyed by step number. We populate it inside the training loop below.

# Snapshot at step 0 (random init) before any training
snapshots[0] = {                                                           # Step-0 snapshot captured BEFORE the loop starts — the truly random state. After this we'll train and snapshot at the listed checkpoints.
    "w1":   net.layer1.weight.detach().clone(),                            # Layer 1's weight tensor. .detach() removes it from the autograd graph; .clone() makes an independent copy so subsequent training doesn't mutate this snapshot.
    "b1":   net.layer1.bias.detach().clone(),                              # Layer 1's bias tensor. Same detach + clone pattern. Without .clone() the dict would hold a reference to the live (training) weights.
    "w2":   net.layer2.weight.detach().clone(),                            # Layer 2's weight tensor. (1, 8) shape — one weight per hidden unit, mapping to the single output.
    "b2":   net.layer2.bias.detach().clone(),                              # Layer 2's bias tensor — a single scalar that gets added at the very end of the forward pass.
    "loss": F.mse_loss(net(x_in), y_target).item(),                        # Compute and store the loss at this checkpoint. Lets us label each snapshot with how good the model was at that moment.
}                                                                          # Close the dict literal.

history = []                                                               # List for the periodic (step, loss) pairs we'll print after training. Separate from snapshots — we sample loss more often than full weight snapshots.
for step in range(1, 2001):                                                # Training loop: 2000 steps total, starting from step 1 (step 0 was the pre-training snapshot above). Adam typically converges in a few hundred steps for this size problem.
    y_pred = net(x_in)                                                     # Forward pass — call the network on the inputs. PyTorch invokes MLP.forward(), which runs Linear → ReLU → Linear and returns predictions.
    loss = F.mse_loss(y_pred, y_target)                                    # Compute the mean-squared-error loss. F.mse_loss is the functional form of nn.MSELoss — it works the same way, just stateless.
    opt.zero_grad(); loss.backward(); opt.step()                           # The three-line training step on one line: clear last step's gradients, backprop through the entire model (Linear → ReLU → Linear), then update all 25 parameters via Adam.
    if step in CHECKPOINTS:                                                # Check whether this step is one of our snapshot points. We snapshot only at chosen steps (saving every step would be wasteful with no extra teaching value).
        snapshots[step] = {                                                # Begin building this checkpoint's snapshot dict (same structure as the step-0 one above).
            "w1":   net.layer1.weight.detach().clone(),                    # Snapshot of layer 1 weights. Will be different from step-0's snapshot because training has nudged the weights.
            "b1":   net.layer1.bias.detach().clone(),                      # Snapshot of layer 1 biases. Each one has been adjusted by gradient descent toward useful values.
            "w2":   net.layer2.weight.detach().clone(),                    # Snapshot of layer 2 weights — how each hidden unit contributes to the output at this point in training.
            "b2":   net.layer2.bias.detach().clone(),                      # Snapshot of layer 2 bias — the final additive offset.
            "loss": loss.item(),                                           # Store the current loss for this checkpoint. .item() pulls the scalar out so the dict holds a plain float, not a 0-dim tensor.
        }                                                                  # Close the snapshot dict for this step.
    if step % 200 == 0:                                                    # Every 200 steps, also append a (step, loss) pair to history. This gives us a coarser loss curve for the printed table below.
        history.append((step, loss.item()))                                # Append (step, loss-as-float). Used for the loss-evolution printout that follows the loop.

print(f"{'step':>5}  {'loss':>8}")                                          # Header row for the loss-over-time table.
print("-" * 16)                                                             # Horizontal divider.
for step, l in history:                                                     # Walk through the captured (step, loss) pairs and print each as one row of the table.
    print(f"{step:>5}  {l:>8.4f}")                                          # Format: right-aligned step (5 chars) and loss (8 chars, 4 decimals). Loss should fall rapidly toward the noise floor.

# Get final predictions
y_pred_nn = net(x_in).squeeze(-1).detach()                                  # Final-model predictions on all 40 inputs. .squeeze(-1) removes the trailing 1-dim axis (from (40,1) back to (40,)). .detach() drops autograd tracking — we just want the values.

## Step 5b — Watch the neural network EVOLVE during training (Karpathy-style)

This is the cool part. We saved snapshots of all the weights at 5 checkpoints during training. Let's visualise how each hidden unit's "knee" moves and how the prediction curve takes shape.

For each checkpoint, we'll show TWO things:

1. **Layer 1 weights as a small grid** — see how each unit's slope and bias change.
2. **Per-unit response curve** — for each of the 8 hidden units, plot `ReLU(w·x + b)` across the input range. Each unit becomes a feature detector for some region of x.

In [ ]:
def render_response_curves(snap, x_range, width=40):                                # Define a helper that turns a model snapshot into ASCII bar plots — one per hidden unit. Width is character count of each bar; snap is one of our captured snapshots.
    """For each hidden unit, render its ReLU output across x_range as a bar."""     # Docstring describing what the function does. Helps a future reader understand the function's intent at a glance.
    w1 = snap["w1"]   # (8, 1)                                                        # Pull out the layer1 weight matrix from the snapshot — it has shape (8, 1), one slope per hidden unit.
    b1 = snap["b1"]   # (8,)                                                          # Pull out the layer1 bias vector — one bias per hidden unit. Combined with w1, this defines each unit's ReLU "kink" function.
    # Compute response per unit, per x point: shape (8, len(x_range))
    x = x_range.unsqueeze(0)                      # (1, len(x))                       # Reshape x_range so we can broadcast against w1. unsqueeze(0) adds a leading dim, making it (1, 40) — ready to broadcast against (8, 1).
    responses = F.relu(w1 * x + b1.unsqueeze(-1))  # (8, len(x))                       # Compute each hidden unit's output for each x. Broadcasting: w1 (8,1) * x (1,40) = (8,40), then add b1 (8,1), apply ReLU. Result shape (8, 40).
    # Normalise each row independently for visual scale
    out = []                                                                           # List that will collect one ASCII bar string per hidden unit. Built up inside the loop.
    for i in range(responses.size(0)):                                                # Iterate over the 8 hidden units. responses.size(0) is 8 — the leading dimension of the tensor we computed above.
        row = responses[i]                                                             # The i-th hidden unit's response across the full x_range — a 1D tensor of length 40 (one response value per x point).
        if row.max().item() < 1e-3:                                                    # If this unit is essentially dead (output near zero across the whole range), draw it as dots. Prevents division by zero in the normalization step below.
            out.append("." * width)                                                    # Dead-unit marker — a row of 40 dots indicates "this unit never fires" in the visible x range.
            continue                                                                    # Skip to the next unit. We've already appended this unit's representation, so no further work needed.
        bars = ""                                                                       # String we'll build up character-by-character — represents the unit's response curve as a bar chart.
        normed = row / row.max()                                                        # Normalize this unit's response to [0, 1] so the tallest point of the curve becomes a full bar. Other units may be much smaller — normalizing keeps them visible.
        for v in normed.tolist():                                                       # Walk through the 40 normalized values, converting each to a character. .tolist() pulls the tensor into a regular Python list for iteration.
            ch = " ▁▂▃▄▅▆▇█"[min(int(v * 8), 8)]                                       # Map a number in [0,1] to one of 9 block characters (space, ▁, ▂, ..., ▇, █). int(v*8) gives 0..8; min protects against v=1.0 giving 8 exactly.
            bars += ch                                                                  # Append the block character to the growing bar string. After 40 iterations the string represents a 40-column bar chart of this unit's response curve.
        out.append(bars)                                                                # Add this unit's completed bar chart to the output list. After the outer loop, out has 8 strings — one per hidden unit.
    return out                                                                          # Return the list of 8 bar-chart strings. The caller will print them with appropriate labels.

# Make a dense x-range for the response plots
x_viz = torch.linspace(-2, 6, 40)                                                       # 40 evenly-spaced x values from -2 to 6, matching the visualization width. Used by render_response_curves to evaluate each hidden unit.

for step in CHECKPOINTS:                                                                # Loop over our 5 snapshot points (0, 100, 500, 1000, 2000) — for each, print a full inspection block showing weights and response curves.
    snap = snapshots[step]                                                              # Retrieve this checkpoint's stored snapshot dict (w1, b1, w2, b2, loss).
    print("=" * 70)                                                                     # Horizontal divider line — 70 equals signs — to visually separate checkpoints from each other.
    print(f"  CHECKPOINT step {step}   (loss = {snap['loss']:.4f})")                    # Print the checkpoint header showing which step this is and how good the model was at that moment.
    print("=" * 70)                                                                     # Closing line of the header banner — mirror of the top line for visual symmetry.
    print()                                                                              # Blank line.
    print(f"  {'unit':<5s}  {'w':>8s}  {'b':>8s}  {'kink at x':>10s}  {'layer2_w':>10s}")  # Column header for the per-unit weight table. Right-align all numeric columns so they line up vertically beneath.
    print("  " + "-" * 50)                                                              # Divider line under the column headers.
    w1 = snap["w1"]; b1 = snap["b1"]; w2 = snap["w2"]                                   # Unpack the tensors we need into local names for brevity. Three statements separated by semicolons on one line.
    for i in range(hidden):                                                              # Iterate over all 8 hidden units and print a row of stats for each.
        wv = w1[i, 0].item()                                                             # The i-th unit's slope. .item() converts the 0-dim tensor to a Python float for clean formatting.
        bv = b1[i].item()                                                                # The i-th unit's bias. Same pattern.
        kink = -bv / wv if abs(wv) > 1e-6 else float('nan')                              # Compute the unit's ReLU kink point analytically. Use NaN sentinel if the slope is essentially zero (would otherwise divide by zero).
        contribution = w2[0, i].item()                                                   # The corresponding entry in layer2's weights — how much this hidden unit's ReLU output contributes (with sign) to the final prediction.
        kink_str = f"{kink:+.2f}" if abs(kink) < 100 else "  inf"                        # Format kink for printing. If it's a huge number (or NaN), show "inf" instead of cluttering the table with massive values.
        print(f"  h_{i+1:<3d}  {wv:>+8.3f}  {bv:>+8.3f}  {kink_str:>10s}  {contribution:>+10.3f}")    # Print this unit's row. ':+' includes the sign, '8.3f' uses 8-char width with 3 decimals, '10s' is a 10-char string field.
    print()                                                                              # Blank line between the weight table and the response curves.
    print(f"  Hidden unit response curves across x = [{-2}, {6}]:")                     # Caption for the bar-chart section.
    curves = render_response_curves(snap, x_viz)                                         # Call our helper to compute the 8 bar-chart strings for this checkpoint's snapshot.
    for i, row in enumerate(curves):                                                     # Print each bar chart on its own line with a unit label.
        print(f"  h_{i+1}: {row}")                                                       # Format: "  h_1: ▁▁▂▃▄▅▆▇█..." — unit name then the bar chart.
    print()                                                                              # Blank line at the end of this checkpoint's block.

**Read the evolution top-to-bottom.**

- **Step 0** — all 8 hidden units have random "knees". Their response curves look noisy. The model knows nothing.
- **Step 100** — units are starting to specialise. Some have learned to respond to negative x, some to positive x.
- **Step 500** — the curves are clearly differentiated. Each unit responds to a particular region.
- **Step 2000 (final)** — beautiful "knee" functions. Each hidden unit covers a different slice of x, and together (weighted by layer 2) they sum to recreate the parabola.

This is what "training" actually looks like inside a neural network: **the weights drift from noise to structure**, where each unit becomes a feature detector for some part of the input. The same thing happens in massive networks like BERT and GPT — just with billions of weights instead of 25.

> 🔑 **The Karpathy moment:** if you watch the response curves develop, you can SEE the model thinking. Each hidden unit is finding a slot — a section of the input it's responsible for. That's machine learning made visible.

### Putting it together: how do 8 knee-shapes become a parabola?

The final prediction is `y = sum(layer2_w[i] * h_i(x)) + layer2_b`. Each hidden unit contributes a piece (positive or negative). The 8 contributions sum to the prediction curve.

In [ ]:
print("Final hidden-unit contributions across the x range:")                            # Header introducing the per-unit decomposition table — the most concrete demonstration of how the network arrives at each prediction.
print("  (Each row is one hidden unit's contribution: layer2_w[i] * ReLU(layer1_w[i]*x + layer1_b[i]))")  # Explanation of what numbers in the table mean. Decomposes the formula into recognisable pieces so the reader connects code → math.
print()                                                                                   # Blank line.
print(f"  {'x':>6s} | " + " ".join(f"h{i+1:>5d}" for i in range(hidden)) + f"  | {'SUM':>7s}  {'TRUTH':>7s}")    # Build the header row. Generator expression "h1, h2, ..., h8" produces one column per hidden unit, then SUM and TRUTH columns.
print("  " + "-" * (10 + hidden*7 + 22))                                                  # Compute exact width of the divider line: 10 chars for x column, 7 per hidden unit (8 of them), 22 for SUM and TRUTH.

final_snap = snapshots[2000]                                                              # Get the final (step 2000) snapshot — the fully-trained weights. We'll use these to compute the decomposition.
w1, b1 = final_snap["w1"], final_snap["b1"]                                               # Unpack layer 1 tensors. w1 is (8,1), b1 is (8,). One slope and one bias per hidden unit.
w2, b2 = final_snap["w2"], final_snap["b2"]                                               # Unpack layer 2 tensors. w2 is (1,8) — eight contribution weights. b2 is (1,) — the final additive offset.

for i in [0, 5, 10, 15, 20, 25, 30, 35, 39]:                                              # Loop over 9 representative inputs spanning the dataset (same indices used in earlier tables for consistency).
    x = x_data[i].item()                                                                  # Pull this input's x as a Python float.
    # Per-unit contribution
    contribs = []                                                                          # List that will hold each hidden unit's contribution to the final prediction for this x.
    for j in range(hidden):                                                                # Loop over all 8 hidden units and compute each one's contribution.
        h_j = max(0.0, w1[j, 0].item() * x + b1[j].item())   # ReLU                       # Compute hidden unit j's activation: ReLU(w * x + b). Use max(0, ...) for ReLU since we're in plain Python here, not PyTorch tensors.
        c   = w2[0, j].item() * h_j                                                        # Multiply by layer2's weight for this unit — that's this unit's signed contribution to the final output.
        contribs.append(c)                                                                  # Save it. After the inner loop, contribs is a list of 8 floats — one per hidden unit.
    total = sum(contribs) + b2.item()                                                       # Sum all 8 contributions and add the layer 2 bias. This is the model's final prediction for this x — equivalent to running net(x).
    truth = y_true[i].item()                                                                # The actual ground-truth value at this x. We'll print it alongside the prediction so the reader can see how close (or off) the model is.
    print(f"  {x:>6.2f} | " + " ".join(f"{c:>+5.1f}" for c in contribs) + f"  | {total:>+7.2f}  {truth:>+7.2f}")  # Print one row of the table. Each contribution is shown with sign, then SUM and TRUTH side by side for easy comparison.
print()                                                                                    # Blank line.
print("Each column is one hidden unit's contribution at each input.")                      # Educational annotation pointing at the table's structure.
print("Adding ACROSS each row reconstructs the prediction. That's all a neural net does.") # Punchline: a neural net is literally a sum of (weighted) hidden-unit activations. No magic — just summing piecewise-linear pieces.

**Loss converges similarly to the polynomial regression** — both end up around the noise floor (~0.25 due to the random noise we added).

The MLP figured out the parabolic shape **without being told to add x² as a feature**. It learned the right features on its own.

## Step 6 — Side-by-side comparison

All three models on the same data:

In [ ]:
print(f"{'x':>6}  {'y_true':>7}  {'linear':>7}  {'poly':>7}  {'NN':>7}")                  # Header row of the four-way comparison table. Right-align each column heading so the numeric rows below align cleanly.
print("-" * 40)                                                                              # Horizontal divider.
for i in [0, 5, 10, 15, 20, 25, 30, 35, 39]:                                                 # Loop over the same 9 representative indices used throughout this notebook for consistency.
    print(f"{x_data[i].item():>6.2f}  {y_true[i].item():>7.2f}  "                            # Print x and the truth. Multi-line string (next 3 print lines continue this same call) for readability.
          f"{y_pred_linear[i].item():>7.2f}  "                                                # Print the linear model's prediction. Should be obviously wrong for the U-shape.
          f"{y_pred_poly[i].item():>7.2f}  "                                                  # Print the polynomial model's prediction. Should be close to y_true.
          f"{y_pred_nn[i].item():>7.2f}")                                                     # Print the neural network's prediction. Also close to y_true. Same data, different model classes.
print()                                                                                       # Blank line.
loss_linear = ((y_pred_linear - y_true) ** 2).mean().item()                                  # Compute the final MSE for the linear model. Same MSE formula as during training: square errors, take mean.
loss_poly   = ((y_pred_poly   - y_true) ** 2).mean().item()                                  # Same for the polynomial regression. Should be much smaller.
loss_nn     = ((y_pred_nn     - y_true) ** 2).mean().item()                                  # Same for the neural network. Should be similar to polynomial — possibly slightly better or worse.
print(f"Final MSE: linear={loss_linear:.3f}   polynomial={loss_poly:.3f}   neural net={loss_nn:.3f}")  # The headline summary: side-by-side MSE for all three models. Linear ~25, polynomial ~1, neural net ~0.2.

**The linear model fails. The polynomial and the neural network both succeed.**

But notice: the polynomial uses **3 parameters**, the neural net uses **25**. The neural net is using extra capacity that it didn't strictly need for this simple problem.

So why ever use a neural network? Because for **complex data** (text, images, banking event sequences), you don't know what the right features are. You can't say "add x²" because the data is too complex for that intuition to apply. A neural network with enough capacity will figure out the features for you.

## Step 7 — Visualise all three models' fits

ASCII overlay so you can see them on the data:

In [ ]:
def ascii_overlay(x, y_true, predictions, markers, width=70, height=18):                # Helper to overlay multiple prediction curves on the same ASCII plot. predictions is a list of tensors; markers is a list of single chars used to draw each.
    xmin, xmax = x.min().item(), x.max().item()                                            # Compute x-axis bounds from the actual data so the plot exactly fills its width.
    all_y = torch.cat([y_true] + [p for p in predictions])                                 # Concatenate the truth tensor with every prediction tensor to compute a SHARED y-axis range — keeps the plots visually comparable.
    ymin, ymax = all_y.min().item(), all_y.max().item()                                    # Use the full range of all_y as plot bounds. If we used per-tensor ranges, the curves would be incomparable.

    grid = [[' '] * width for _ in range(height)]                                          # Build a blank 2D character grid we'll fill in below. List of lists so we can mutate individual cells.
    # Plot truth as '.'
    for xi, yi in zip(x.tolist(), y_true.tolist()):                                        # Loop through each (x, y_true) pair and place a '·' character at the corresponding grid position.
        col = int((xi - xmin) / (xmax - xmin) * (width - 1))                                # Same coordinate-conversion math as ascii_scatter — normalise x to [0,1] then scale to grid width.
        row = height - 1 - int((yi - ymin) / (ymax - ymin) * (height - 1))                   # Flip y because text-grid rows count downward but y values count upward. Higher y → lower row index.
        if 0 <= col < width and 0 <= row < height:                                          # Bounds check — protects against rounding edge cases at the plot boundaries.
            grid[row][col] = '·'                                                            # Place the truth marker. We use middot ('·') for the underlying data so the prediction letters stand out.
    # Plot each prediction
    for pred, marker in zip(predictions, markers):                                          # Loop over each (predictions tensor, single-char marker) pair. Process linear, then polynomial, then neural net.
        for xi, yi in zip(x.tolist(), pred.tolist()):                                       # Inner loop: for each predicted (x, y), place the marker at the corresponding grid cell.
            col = int((xi - xmin) / (xmax - xmin) * (width - 1))                            # Same x-coord transform as above.
            row = height - 1 - int((yi - ymin) / (ymax - ymin) * (height - 1))               # Same y-coord transform.
            if 0 <= col < width and 0 <= row < height and grid[row][col] == ' ':           # Bounds check + only-write-if-empty (so we don't overwrite truth dots or earlier prediction markers).
                grid[row][col] = marker                                                     # Stamp the prediction marker for this curve at the computed grid position.

    print(f"{ymax:6.1f} |" + "".join(grid[0]))                                              # Print the top row of the grid prefixed with the maximum y value as a label.
    for row in grid[1:-1]:                                                                   # Print the middle rows. grid[1:-1] excludes first and last (which get y labels).
        print("       |" + "".join(row))                                                    # Each middle row gets a 7-space prefix to align with the labeled top/bottom rows.
    print(f"{ymin:6.1f} |" + "".join(grid[-1]))                                              # Print the bottom row of the grid with the minimum y value as a label.
    print("       +" + "-" * width)                                                          # X-axis line: '+' at the corner, then `width` dashes.
    print(f"       Legend:  ·  = true data  L = linear  P = polynomial  N = neural net")  # Print a legend so readers know what each character represents.

print("All three models' predictions overlaid on the data:")                                # Caption introducing the overlay plot.
ascii_overlay(x_data, y_true, [y_pred_linear, y_pred_poly, y_pred_nn], ['L', 'P', 'N'])      # Draw the overlay. Passing all three prediction tensors and one marker character each. Order matters — the LAST one drawn shows up on top in case of overlap.

**Read the plot:**
- The `·` dots are the true data (the parabola).
- The `L`s trace a straight line — linear regression can ONLY draw lines.
- The `P`s follow the parabola — polynomial regression nails it.
- The `N`s also follow the parabola — neural network also nails it.

Both `P` and `N` succeed. The `L` is doomed.

## Step 8 — The big reveal: what IS BERT, then?

Now we can answer the question that may have been confusing.

**BERT, GPT, PRAGMA — they are NEURAL NETWORKS.** Specifically, a kind called **Transformers**.

A Transformer is built out of:

```
input tokens
   │
   ▼
Embedding      ─►  linear lookup (one of the only LINEAR-only parts)
   │
   ▼
Attention      ─►  multiple LINEAR projections (Q, K, V) + softmax (NONLINEAR) + matmul
   │
   ▼
Feed-forward   ─►  Linear → GELU (NONLINEAR) → Linear     ← this is just an MLP!
   │
   ▼
(repeat)
   │
   ▼
Output head    ─►  Linear projection to vocabulary scores
```

The key part: there are **nonlinearities** sprinkled throughout (softmax inside attention, GELU inside the feed-forward sub-layer). Without them, the whole thing would collapse to one big linear function, just like stacking two Linear layers without a ReLU collapses to a single Linear.

**So no — BERT is not linear regression. BERT is a neural network with attention layers.**

What's the same:
- ✅ The 5-line training loop (predict → loss → backward → step)
- ✅ Gradient descent
- ✅ Embeddings as the input layer
- ✅ A Linear "head" at the output

What's different:
- ❌ The middle of the model: linear regression has `w*x + b`; BERT has stacked attention + feed-forward layers with nonlinearities.

Same training. Different model.

## Step 9 — The complete picture: model family tree

| Model | Architecture | Parameters | Where it shines |
|---|---|---|---|
| **Linear regression** (L1) | `y = w*x + b` | 2 | Predicting a single number from a single number, when the relationship is linear |
| **Polynomial regression** (L1.5 above) | `y = w₁x + w₂x² + ... + b` | 3 to N | Same, when relationship is polynomial and you know what features to add |
| **MLP / Neural net** (L1.5 above) | Linear → ReLU → Linear → (...) | tens to thousands | Predicting any function of any feature vector, when the features can be learned |
| **CNN** (not in this course) | Convolutions + ReLU + pooling | millions | Images |
| **RNN / LSTM** (briefly in L3b) | Recurrent steps over a sequence | millions | Sequences (but with memory issues — see L3b) |
| **Transformer** (L4, L5, BERT, PRAGMA, GPT) | Attention + feed-forward + LayerNorm + residual | millions to **trillions** | Sequences, where every token can look at every other token |

**Every single one is trained with the SAME 5-line loop.** Only the model class changes.

## Step 10 — Inspect what the neural net "learned"

Just for fun: peek inside the trained MLP and see what each hidden unit responds to.

In [ ]:
# Run the trained MLP and look at the hidden activations
with torch.no_grad():                                                                       # Context manager that disables autograd inside this block. Saves memory and time — we're inspecting, not training.
    hidden = F.relu(net.layer1(x_in))   # (40, 8)                                          # Manually compute layer1's pre-output, then apply ReLU — gives us the actual hidden activations the network feeds to layer2. Shape (40, 8).

print("Hidden unit activations across the input range:")                                    # Caption introducing the activation map.
print(f"  Each row = one input x; each column = one of the 8 hidden units")                  # Explain the table's structure — important so the reader knows what they're looking at.
print(f"  '·' means 0 (ReLU clipped it); '█' means active")                                  # Legend for the visual encoding. ReLU's job is exactly this — turn negative things into zero.
print()                                                                                      # Blank line.
print(f"  {'x':>6s} | " + " ".join(f"h{i}" for i in range(8)))                                # Header row. Generator expression produces "h0 h1 h2 ... h7" for the column headers.
print("  " + "-" * 30)                                                                       # Horizontal divider.
for i in [0, 5, 10, 15, 20, 25, 30, 35, 39]:                                                 # Loop over 9 representative inputs spanning the dataset (consistent with other tables).
    row = ""                                                                                  # String we'll build up — one character per hidden unit's activation state for this input.
    for h in hidden[i].tolist():                                                             # Walk through this input's 8 hidden activations.
        row += " " + ("█" if h > 0.5 else ("▄" if h > 0.01 else "·"))                        # Map activation magnitude to one of 3 characters: '█' for clearly active, '▄' for slightly active, '·' for ReLU-clipped (zero).
    print(f"  {x_data[i].item():>6.2f} |{row}")                                              # Print this row: x value, the divider '|', and the 8-character activation pattern.
print()                                                                                      # Blank line.
print("Each hidden unit has learned to respond to a different part of the x range.")        # Educational annotation #1: the trained units have specialised.
print("Some fire for negative x, some for positive x, some for the middle.")                 # Annotation #2: this is what "feature detection" looks like — each unit owns a slice of x.
print("Adding them up (with the weights learned in layer2) reconstructs the parabola.")     # Annotation #3: the final prediction is built by summing these specialised pieces. Same insight as Step 6 above.

## Step 11 — Things to try

### 🟢 Easy

1. **Different curves.** Replace `y_true = x² - 4x + 3` with `y_true = torch.sin(x_data)` (or any other curve you want). Re-train all three models. Polynomial regression will fail unless you add more features; the neural network will succeed automatically.

2. **More hidden units.** Change `MLP(hidden=8)` to `MLP(hidden=64)`. Does it train better? Worse? Faster? Slower?

### 🟡 Medium

3. **Deeper network.** Add another hidden layer:
   ```python
   self.layer1 = nn.Linear(1, 8)
   self.layer2 = nn.Linear(8, 8)
   self.layer3 = nn.Linear(8, 1)
   def forward(self, x):
       h = F.relu(self.layer1(x))
       h = F.relu(self.layer2(h))
       return self.layer3(h)
   ```
   What does deeper buy you? (On this simple problem, not much. On real problems, depth is essential.)

4. **Replace ReLU with Sigmoid or Tanh.** Try `torch.sigmoid` or `torch.tanh` instead of `F.relu`. Compare training curves.

### 🔴 Hard

5. **What happens with NO activation function?** Remove the `F.relu` line. Now the MLP collapses to a single linear function. Train it and verify it does no better than linear regression — this is the "without nonlinearity, depth doesn't help" lesson.

6. **A "neural network" with attention.** Replace the MLP's hidden layer with a `nn.MultiheadAttention` layer. (You'll need to reshape your input to `(batch, seq=1, features)`.) This is the smallest possible Transformer-like model. Train it and verify it can also fit the parabola — though attention is wildly overkill for a 1D scalar regression.


## Summary

You now know:

- ✅ Linear regression fits **lines** (only).
- ✅ Polynomial regression (still "linear regression" mathematically) fits **polynomials** if you handcraft x² and x³ as features.
- ✅ A neural network with a nonlinearity (ReLU, GELU, etc.) can fit **any smooth function** without needing handcrafted features — it learns them.
- ✅ **BERT, PRAGMA, GPT are neural networks** (specifically Transformers). Their power comes from learnable embeddings + many attention layers + nonlinearities. They are NOT linear regression.
- ✅ The **5-line training loop is the same** for all of these. What changes is the model class.

Open [Lesson 4](lesson_04_tiny_bert.ipynb) again with fresh eyes — that tiny BERT is a neural network with about 3000 parameters that uses attention. Everything before that lesson was warm-up.
